# vCRGlove — Clinical Movement Analysis Dashboard

This notebook loads the CSV exports produced by the vCRGlove iPhone app and generates:
- **Per-session metrics summary** (cycle count, frequency, amplitude, decrement, rhythm, pauses)
- **Longitudinal trend plots** per task and hand (pre vs. post stimulation)
- **Raw signal plots** for individual trials
- **Group comparison table** exportable to Excel

---
### Setup
```
pip install pandas matplotlib seaborn scipy openpyxl
```
Place the exported files next to this notebook:
- `metrics-YYYYMMDD-HHmmss.csv` (one row per trial)
- `samples-YYYYMMDD-HHmmss.csv` (one row per raw signal sample) — optional

Or edit the `METRICS_FILE` / `SAMPLES_FILE` paths below.

In [ ]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# ── File discovery ──────────────────────────────────────────────────────────
# Looks for the most-recent export in the same directory as this notebook.
# Override with absolute paths if needed.

def _latest(pattern: str) -> str | None:
    files = sorted(glob.glob(pattern))
    return files[-1] if files else None

METRICS_FILE = _latest("metrics-*.csv") or "metrics.csv"
SAMPLES_FILE = _latest("samples-*.csv") or "samples.csv"

print(f"Metrics : {METRICS_FILE}")
print(f"Samples : {SAMPLES_FILE}")

In [ ]:
# ── Load metrics ─────────────────────────────────────────────────────────────

df = pd.read_csv(METRICS_FILE, parse_dates=["session_date", "started_at"])

# Friendly display names
TASK_LABELS = {
    "3.4": "Finger Tapping (3.4)",
    "3.5": "Hand Open/Close (3.5)",
    "3.6": "Pronation/Supination (3.6)",
}
CONTEXT_ORDER = ["baseline", "preStim", "postStim", "unspecified"]

df["task_label"] = df["task"].map(TASK_LABELS).fillna(df["task"])
df["context"] = pd.Categorical(df["context"], categories=CONTEXT_ORDER, ordered=True)
df["date"] = df["session_date"].dt.date

print(f"Loaded {len(df)} trials from {df['patient_id'].nunique()} patient(s), "
      f"{df['session_id'].nunique()} session(s)")
df.head(3)

## 1 · Session Overview

In [ ]:
METRICS = [
    ("cycle_count",             "Cycle count",              ""),
    ("frequency_hz",            "Frequency (Hz)",           "Hz"),
    ("mean_amplitude",          "Mean amplitude",           "a.u."),
    ("amplitude_decrement_slope","Ampl. decrement slope",  "/cycle"),
    ("rhythm_cv",               "Rhythm variability (CV)",  ""),
    ("pause_count",             "Pause count",              ""),
    ("onset_latency_sec",       "Onset latency (s)",        "s"),
    ("quality_index",           "Quality index (0–1)",      ""),
]

summary = (
    df.groupby(["patient_id", "task_label", "side", "context"])[
        [m for m, *_ in METRICS]
    ].mean().round(3)
)
summary

## 2 · Longitudinal Trends (per patient)

In [ ]:
PATIENT_ID = df["patient_id"].iloc[0]   # ← change to target patient
SIDE       = "right"                     # ← "left" or "right"

pdf = df[(df["patient_id"] == PATIENT_ID) & (df["side"] == SIDE)].copy()

tasks = pdf["task"].unique()
plot_metrics = [
    ("frequency_hz",   "Frequency (Hz)"),
    ("mean_amplitude", "Mean Amplitude"),
    ("rhythm_cv",      "Rhythm CV"),
    ("pause_count",    "Pauses"),
]

context_colors = {
    "baseline":    "steelblue",
    "preStim":     "darkorange",
    "postStim":    "seagreen",
    "unspecified": "slategray",
}

fig, axes = plt.subplots(
    len(plot_metrics), len(tasks),
    figsize=(5 * len(tasks), 3.5 * len(plot_metrics)),
    sharex="col", sharey="row"
)
axes = axes.reshape(len(plot_metrics), len(tasks))

for col, task in enumerate(tasks):
    tdf = pdf[pdf["task"] == task].sort_values("session_date")
    for row, (metric, ylabel) in enumerate(plot_metrics):
        ax = axes[row, col]
        for ctx, grp in tdf.groupby("context", observed=True):
            ax.scatter(
                grp["session_date"], grp[metric],
                label=ctx, color=context_colors.get(str(ctx), "gray"),
                s=60, zorder=3
            )
            if len(grp) > 1:
                ax.plot(
                    grp["session_date"], grp[metric],
                    color=context_colors.get(str(ctx), "gray"),
                    alpha=0.4, linewidth=1
                )
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
        if row == 0:
            ax.set_title(TASK_LABELS.get(task, task), fontweight="bold")
        if col == 0:
            ax.set_ylabel(ylabel)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title="Context", loc="upper right", bbox_to_anchor=(1.01, 1))
fig.suptitle(f"Patient: {PATIENT_ID} · Hand: {SIDE}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"trend_{PATIENT_ID}_{SIDE}.pdf", bbox_inches="tight")
plt.show()
print("Saved trend PDF.")

## 3 · Pre vs. Post Stimulation Comparison

In [ ]:
pre_post = df[df["context"].isin(["preStim", "postStim"])].copy()

if pre_post.empty:
    print("No preStim/postStim data found.")
else:
    fig, axes = plt.subplots(1, len(tasks), figsize=(5 * len(tasks), 4), sharey=False)
    if len(tasks) == 1:
        axes = [axes]
    for ax, task in zip(axes, tasks):
        tdf = pre_post[pre_post["task"] == task]
        sns.boxplot(
            data=tdf, x="context", y="frequency_hz", hue="side",
            order=["preStim", "postStim"], ax=ax,
            palette={"left": "#4e79a7", "right": "#f28e2b"}
        )
        ax.set_title(TASK_LABELS.get(task, task), fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Frequency (Hz)")

        # Wilcoxon signed-rank (paired) if same sessions have both measurements
        paired = tdf.pivot_table(
            index=["session_id", "side"], columns="context",
            values="frequency_hz", aggfunc="mean"
        ).dropna()
        if len(paired) >= 5:
            stat, p = stats.wilcoxon(paired["preStim"], paired["postStim"])
            ax.set_title(f"{TASK_LABELS.get(task, task)}\nWilcoxon p={p:.3f}",
                         fontweight="bold")

    plt.suptitle("Frequency: Pre vs. Post Stimulation", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("pre_post_comparison.pdf", bbox_inches="tight")
    plt.show()

## 4 · Raw Signal Plot (single trial)

In [ ]:
if not os.path.exists(SAMPLES_FILE):
    print(f"No samples file found at '{SAMPLES_FILE}'. Skip this cell or export samples from the app.")
else:
    sdf = pd.read_csv(SAMPLES_FILE)

    # Pick the first trial that has data
    TRIAL_ID = sdf["trial_id"].iloc[0]   # ← paste any trial_id from the metrics table

    trial_samples = sdf[sdf["trial_id"] == TRIAL_ID]
    trial_meta    = df[df["trial_id"] == TRIAL_ID]

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(trial_samples["t_sec"], trial_samples["value"],
            linewidth=1.2, color="steelblue", label="Raw signal")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Signal value (a.u.)")
    if not trial_meta.empty:
        m = trial_meta.iloc[0]
        ax.set_title(
            f"Trial {TRIAL_ID[:8]}… | "
            f"{TASK_LABELS.get(m['task'], m['task'])} · {m['side']} · "
            f"{m['cycle_count']} cycles @ {m['frequency_hz']:.2f} Hz"
        )
    plt.tight_layout()
    plt.savefig(f"signal_{TRIAL_ID[:8]}.pdf", bbox_inches="tight")
    plt.show()

## 5 · Export to Excel (clinic hand-off)

In [ ]:
EXCEL_OUT = "vCRGlove_export.xlsx"

with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl") as writer:
    # Sheet 1: all trials, human-readable
    export_df = df[[
        "patient_id", "session_date", "context",
        "task_label", "side", "source",
        "cycle_count", "frequency_hz", "mean_amplitude",
        "amplitude_decrement_slope", "rhythm_cv",
        "pause_count", "onset_latency_sec", "quality_index",
        "duration_sec", "sample_count"
    ]].rename(columns={
        "task_label": "task",
        "amplitude_decrement_slope": "ampl_decrement",
        "onset_latency_sec": "onset_latency_s",
        "sample_count": "n_samples",
    })
    export_df.to_excel(writer, sheet_name="Trials", index=False)

    # Sheet 2: session-level averages
    session_avg = (
        df.groupby(["patient_id", "date", "context", "task_label", "side"])[
            ["cycle_count", "frequency_hz", "mean_amplitude",
             "rhythm_cv", "pause_count", "quality_index"]
        ].mean().round(3).reset_index()
    )
    session_avg.to_excel(writer, sheet_name="Session averages", index=False)

    # Sheet 3: pre/post delta per session
    if not pre_post.empty:
        delta = pre_post.pivot_table(
            index=["patient_id", "session_id", "task_label", "side"],
            columns="context",
            values=["frequency_hz", "mean_amplitude", "rhythm_cv", "pause_count"],
            aggfunc="mean"
        )
        delta.columns = ["_".join(c) for c in delta.columns]
        for metric in ["frequency_hz", "mean_amplitude", "rhythm_cv", "pause_count"]:
            pre_col  = f"{metric}_preStim"
            post_col = f"{metric}_postStim"
            if pre_col in delta.columns and post_col in delta.columns:
                delta[f"{metric}_delta"] = delta[post_col] - delta[pre_col]
        delta.reset_index().to_excel(writer, sheet_name="Pre-Post Delta", index=False)

print(f"Saved: {EXCEL_OUT}")

---
## Workflow for the clinic

1. **On the iPhone** — open *Movement Test → (⋯) → Export → CSV — metrics per trial* and share the file via AirDrop or Mail to the clinical Mac.
2. **On the Mac** — copy the `metrics-*.csv` (and optionally `samples-*.csv`) into the same folder as this notebook.
3. **Run all cells** (Kernel → Restart & Run All).
4. Open `vCRGlove_export.xlsx` in Excel / Numbers for further analysis, or use the saved PDF plots directly.

> **Privacy note**: The app stores only a pseudonymized `patient_id`, never names or DOB. Ensure the ID mapping table is stored separately and securely.